# K-IDS: Decision Tree Training

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from imblearn.over_sampling import SMOTE

## 2. Load Preprocessed Datasets

In [ ]:
LABEL_NAMES = {
    0: 'Botnet (Mirai)',
    1: 'Miner (Kinsing/TeamTNT/XMRig)',
    2: 'Trojan (Agent)',
    3: 'Trojan (Coinminer)',
    4: 'Benign (MariaDB)',
    5: 'Benign (Data-Caching)',
    6: 'Benign (Media-Streaming)',
    7: 'Benign (Web-Serving)',
}

syscall_df = pd.read_csv('../feature_engineering/syscall_dataset.csv')
flow_df    = pd.read_csv('../feature_engineering/network_flow_dataset.csv')

print(f'Syscall dataset : {syscall_df.shape}')
print(f'Network flow    : {flow_df.shape}')

## 3. Prepare Features

In [ ]:
# Syscall: pakai 5-gram columns
X_syscall = syscall_df[['n1', 'n2', 'n3', 'n4', 'n5']]
y_syscall = syscall_df['label']

# Network flow: semua kolom kecuali label
X_flow = flow_df.drop(columns=['label'])
y_flow = flow_df['label']

print('Syscall features :', X_syscall.columns.tolist())
print('Flow features    :', X_flow.columns.tolist())
print('\nSyscall label distribution:')
print(y_syscall.map(LABEL_NAMES).value_counts())
print('\nFlow label distribution:')
print(y_flow.map(LABEL_NAMES).value_counts())

## 4. Handle Class Imbalance (SMOTE)

In [ ]:
# Syscall
sm = SMOTE(random_state=42, k_neighbors=min(3, y_syscall.value_counts().min() - 1))
X_syscall_res, y_syscall_res = sm.fit_resample(X_syscall, y_syscall)
print('Syscall after SMOTE:', pd.Series(y_syscall_res).map(LABEL_NAMES).value_counts())

# Network flow
sm2 = SMOTE(random_state=42, k_neighbors=min(3, y_flow.value_counts().min() - 1))
X_flow_res, y_flow_res = sm2.fit_resample(X_flow, y_flow)
print('\nFlow after SMOTE:', pd.Series(y_flow_res).map(LABEL_NAMES).value_counts())

## 5. Train/Test Split

In [ ]:
X_train_sys, X_test_sys, y_train_sys, y_test_sys = train_test_split(
    X_syscall_res, y_syscall_res, test_size=0.2, random_state=42, stratify=y_syscall_res
)

X_train_flow, X_test_flow, y_train_flow, y_test_flow = train_test_split(
    X_flow_res, y_flow_res, test_size=0.2, random_state=42, stratify=y_flow_res
)

print(f'Syscall  - Train: {len(X_train_sys)}, Test: {len(X_test_sys)}')
print(f'Flow     - Train: {len(X_train_flow)}, Test: {len(X_test_flow)}')

## 6. Train Decision Tree

In [ ]:
# Syscall model
dt_syscall = DecisionTreeClassifier(max_depth=10, min_samples_leaf=2, random_state=42)
dt_syscall.fit(X_train_sys, y_train_sys)

# Network flow model
dt_flow = DecisionTreeClassifier(max_depth=10, min_samples_leaf=2, random_state=42)
dt_flow.fit(X_train_flow, y_train_flow)

print('Models trained!')

## 7. Evaluation

In [ ]:
def evaluate(model, X_test, y_test, label_names, title):
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    labels = sorted(y_test.unique())
    names  = [label_names.get(l, str(l)) for l in labels]

    print(f'=== {title} ===')
    print(f'Accuracy: {acc:.4f}')
    print(classification_report(y_test, y_pred, target_names=names))

    cm = confusion_matrix(y_test, y_pred, labels=labels)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=names, yticklabels=names)
    plt.title(f'Confusion Matrix - {title}')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.tight_layout()
    plt.savefig(f'../feature_engineering/output/cm_{title.lower().replace(" ", "_")}.png', dpi=150)
    plt.show()

evaluate(dt_syscall, X_test_sys,  y_test_sys,  LABEL_NAMES, 'Syscall')
evaluate(dt_flow,    X_test_flow, y_test_flow, LABEL_NAMES, 'Network Flow')

## 8. Cross-Validation

In [ ]:
cv_sys  = cross_val_score(dt_syscall, X_syscall_res, y_syscall_res, cv=5, scoring='accuracy')
cv_flow = cross_val_score(dt_flow,    X_flow_res,    y_flow_res,    cv=5, scoring='accuracy')

print(f'Syscall  CV Accuracy: {cv_sys.mean():.4f} ± {cv_sys.std():.4f}')
print(f'Flow     CV Accuracy: {cv_flow.mean():.4f} ± {cv_flow.std():.4f}')

## 9. Visualize Tree (Syscall)

In [ ]:
labels = sorted(y_syscall_res.unique())
names  = [LABEL_NAMES.get(l, str(l)) for l in labels]

plt.figure(figsize=(24, 10))
plot_tree(dt_syscall, feature_names=X_syscall.columns, class_names=names,
          filled=True, max_depth=3, fontsize=8)
plt.title('Decision Tree - Syscall (depth=3 preview)')
plt.tight_layout()
plt.savefig('../feature_engineering/output/tree_syscall.png', dpi=100)
plt.show()

## 10. Save Models

In [ ]:
import os
os.makedirs('../models', exist_ok=True)

joblib.dump(dt_syscall, '../models/dt_syscall.pkl')
joblib.dump(dt_flow,    '../models/dt_flow.pkl')

print('Models saved:')
print('  ../models/dt_syscall.pkl')
print('  ../models/dt_flow.pkl')